# 03 — `alexandria-client`: Cryptographic Sync, Dynamic Rules, `:explain` & Passive Audit

This notebook explores **Part IV (`C-CLI`)** of Specification v0.3.0:
1. **Zero-Hub Hot Path & Cryptographic Sync (`S1`–`S13`, `V0`–`V4`)**: `alexandria-client` polls `/v1/namespaces/demo/versions` over HTTPS (`-hub-ca`), verifies `eddsa-jcs-2022` proofs against `trust.hub_did`, verifies `ni:///sha-256;` digests, and swaps an atomic in-memory snapshot.
2. **Admin Observability**: `/healthz`, `/readyz`, `/status`, `/versions`, and Prometheus `/metrics`.
3. **Dynamic Rule Set (`Appendix A.6`)**: Seeded from `deploy/rules.seed.json`, inspected and mutated live via `GET /rules`, `PUT /rules/{id}`, `POST /rules`, `DELETE /rules/{id}` with `ETag`/`If-Match`, plus side-effect-free `POST /rules:explain`.
4. **Enforcement Modes (`enforce`, `shadow`, `observe`) & `@context` Isolation (`CONTEXT_NOT_IN_PACKAGE`)**.
5. **Passive Catalog Audit (`§4.11`)**: Buffering upstream catalog responses and predicting `AUDIT_WILL_WARN` / `AUDIT_WILL_REJECT` when a new breaking version is published!

In [ ]:
from lab import AlexandriaLab, show_events_table, show_state_machine

lab = AlexandriaLab(namespace="demo", compiler="poc", initial_clock="2026-09-01T10:00:00Z")
lab.start_hub_stack()

ana = lab.token("ana", admin=True)
sergio = lab.token("sergio")
lab.request("POST", f"{lab.hub_url}/v1/namespaces", token=ana, body={"slug": "demo"}, verbose=False)
lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/stewards", token=ana, body={"iss": lab.idp_url, "sub": "sergio"}, verbose=False)

draft_100 = {
    "version": "1.0.0",
    "profiles": [f"{lab.hub_url}/pub/demo/profiles/odrl/1.0.0"],
    "slots": [
        {"slot": "policy", "obligation": "mandatory", "applies_to": ["policy_intake"]},
        {"slot": "catalog", "obligation": "mandatory", "applies_to": ["catalog_entry"]},
    ],
    "files": [
        {
            "path": "schemas/policy.schema.json",
            "media_type": "application/schema+json",
            "role": "json-schema",
            "slot": "policy",
            "applies_to": ["policy_intake"],
            "content": {
                "type": "object",
                "properties": {"action": {"enum": ["odrl:use", "ex:access"]}},
                "required": ["action"],
            },
        },
        {
            "path": "schemas/catalog.schema.json",
            "media_type": "application/schema+json",
            "role": "json-schema",
            "slot": "catalog",
            "applies_to": ["catalog_entry"],
            "content": {"type": "object"},
        },
    ],
}
_, hdr_d, _ = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/drafts", token=sergio, body=draft_100, verbose=False)
draft_id = hdr_d["location"].rsplit("/", 1)[-1]
_, _, cand = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/drafts/{draft_id}:propose", token=sergio, verbose=False)
cand_id = cand["id"].rsplit("/", 1)[-1]
lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/candidates/{cand_id}:approve", token=sergio, verbose=False)

# Advance clock so 1.0.0 is in force (past P2D propagation_window) and start alexandria-client
lab.set_clock("2026-09-04T10:00:00Z")
lab.start_client()
status_doc = lab.wait_synced("2026-09-01T10:00:00Z")
show_state_machine(status_doc["state"], subtitle=f"Listing seq={status_doc['last_accepted_listing']['sequence']}")

## 1. Admin Listener: `/status`, `/versions`, `/readyz` & `/metrics`

In [ ]:
st, _, _ = lab.request("GET", f"{lab.admin_url}/readyz")
assert st == 200

st, _, st_body = lab.request("GET", f"{lab.admin_url}/status")
assert st == 200 and st_body["state"] == "Active"

st, _, vers = lab.request("GET", f"{lab.admin_url}/versions")
assert st == 200 and vers[0]["version"] == "1.0.0"

st, _, prom_text = lab.request("GET", f"{lab.admin_url}/metrics")
assert st == 200 and "vhub_" in prom_text

## 2. Dynamic Rule Engine (`GET /rules`, `POST /rules`, `PUT /rules/{id}`, `DELETE /rules/{id}` & `POST /rules:explain`)

All rule mutations require `If-Match` matching the current rule-set revision `ETag`. `POST /rules:explain` evaluates any hypothetical request against the active snapshot and rule revision **with zero side effects** (no upstream forwarding, no agreement capture, no events emitted).

In [ ]:
st, hdr_rules, ruleset = lab.request("GET", f"{lab.admin_url}/rules")
assert st == 200 and ruleset["revision"] == 1
rev_etag = hdr_rules["etag"]
print(f"Seeded rules count: {len(ruleset['rules'])}, ETag: {rev_etag}")

# Dry-run simulation with POST /rules:explain
st, _, exp_bad = lab.request(
    "POST", f"{lab.admin_url}/rules:explain",
    body={
        "listener": "dsp-ingress",
        "method": "POST",
        "path": "/negotiations/urn:uuid:p-1/request",
        "body": {"@type": "ContractRequestMessage", "providerPid": "urn:uuid:p-1", "consumerPid": "urn:uuid:c-1", "action": "odrl:read"},
    },
)
assert st == 200 and exp_bad["rule"] == "contract-request" and exp_bad["action"] == "reject"

# Add a custom rule (POST /rules) and remove it (DELETE /rules/{id}) with optimistic concurrency
custom_rule = {
    "id": "custom-health-ping",
    "priority": 950,
    "listener": "dsp-ingress",
    "match": {"method": "POST", "path": "/custom/ping"},
    "message": "ContractNegotiationEventMessage",
    "verify": {"gates": ["dsp_message"]},
    "mode": "observe",
}
st, hdr_r2, rs2 = lab.request("POST", f"{lab.admin_url}/rules", headers={"If-Match": rev_etag}, body=custom_rule)
assert st == 201 and rs2["revision"] == 2

st, hdr_r3, _ = lab.request("DELETE", f"{lab.admin_url}/rules/custom-health-ping", headers={"If-Match": hdr_r2["etag"]})
assert st == 204

## 3. DSP Hot-Path Modes (`enforce` vs `shadow`) & `@context` Isolation (`CONTEXT_NOT_IN_PACKAGE`)

Let's send real DSP messages through `dsp-ingress`:
1. `odrl:use` conforms to `1.0.0` → `200 OK` forwarded to `fake-connector`.
2. `odrl:read` violates `1.0.0` → in `enforce` mode, rejected with `400 VERDICT_VIOLATION` and never reaches `fake-connector`.
3. Switching `contract-request` to `shadow` mode forwards `odrl:read` (`200 OK`, no `VHub-Verdict` header) while recording a `vhub.verdict` violation in `events.jsonl`!
4. Restoring `enforce` and sending an external `@context` (`https://evil.example/ctx.jsonld`) triggers `400 CONTEXT_NOT_IN_PACKAGE`.

In [ ]:
msg_valid = {"@type": "ContractRequestMessage", "providerPid": "urn:uuid:p-1", "consumerPid": "urn:uuid:c-1", "action": "odrl:use"}
msg_invalid = {"@type": "ContractRequestMessage", "providerPid": "urn:uuid:p-1", "consumerPid": "urn:uuid:c-1", "action": "odrl:read"}

st, _, _ = lab.request("POST", f"{lab.dsp_ingress_url}/negotiations/urn:uuid:p-1/request", body=msg_valid)
assert st == 200

st, hdr_v, err_v = lab.request("POST", f"{lab.dsp_ingress_url}/negotiations/urn:uuid:p-1/request", body=msg_invalid)
assert st == 400 and err_v["code"] == "VERDICT_VIOLATION"
assert 'code="VERDICT_VIOLATION"' in hdr_v["vhub-verdict"]

# Switch contract-request to shadow mode
_, hdr_cr, cr_rule = lab.request("GET", f"{lab.admin_url}/rules/contract-request", verbose=False)
cr_rule["mode"] = "shadow"
st, hdr_cr2, _ = lab.request("PUT", f"{lab.admin_url}/rules/contract-request", headers={"If-Match": hdr_cr["etag"]}, body=cr_rule)
assert st == 200

st, hdr_sh, _ = lab.request("POST", f"{lab.dsp_ingress_url}/negotiations/urn:uuid:p-1/request", body=msg_invalid)
assert st == 200 and "vhub-verdict" not in hdr_sh

# Restore enforce mode & test CONTEXT_NOT_IN_PACKAGE
cr_rule["mode"] = "enforce"
lab.request("PUT", f"{lab.admin_url}/rules/contract-request", headers={"If-Match": hdr_cr2["etag"]}, body=cr_rule, verbose=False)

st, _, err_ctx = lab.request(
    "POST", f"{lab.dsp_ingress_url}/negotiations/urn:uuid:p-1/request",
    body={"@context": ["https://evil.example/ctx.jsonld"], "@type": "ContractRequestMessage", "action": "odrl:use"},
)
assert st == 400 and err_ctx["code"] == "CONTEXT_NOT_IN_PACKAGE"

## 4. Passive Catalog Audit (`GET /audit/reports/latest`)

When a `CatalogRequestMessage` flows through `dsp-ingress`, `alexandria-client` buffers the upstream connector's catalog response in memory without blocking traffic. When a breaking version `2.0.0` is later published requiring a `datasetTitle` field on `catalog_entry`, the client automatically re-evaluates buffered catalog responses at `new_content_from(2.0.0)` (`AUDIT_WILL_WARN`) and `enforced_from(2.0.0)` (`AUDIT_WILL_REJECT`)!

In [ ]:
try:
    # 1. Trigger a catalog request so Audit.Observe buffers fake-connector's response
    st, _, _ = lab.request("POST", f"{lab.dsp_ingress_url}/catalog/request", body={"@type": "CatalogRequestMessage"})
    assert st == 200

    # 2. Publish breaking version 2.0.0 tightening catalog_entry schema
    lab.set_clock("2026-09-10T10:00:00Z")
    draft_200 = {
        "version": "2.0.0",
        "profiles": [f"{lab.hub_url}/pub/demo/profiles/odrl/2.0.0"],
        "slots": [
            {"slot": "policy", "obligation": "mandatory", "applies_to": ["policy_intake"]},
            {"slot": "catalog", "obligation": "mandatory", "applies_to": ["catalog_entry"]},
        ],
        "files": [
            draft_100["files"][0],
            {
                "path": "schemas/catalog.schema.json",
                "media_type": "application/schema+json",
                "role": "json-schema",
                "slot": "catalog",
                "applies_to": ["catalog_entry"],
                "content": {"type": "object", "required": ["datasetTitle"]},
            },
        ],
    }
    _, hdr_d2, _ = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/drafts", token=sergio, body=draft_200, verbose=False)
    d2_id = hdr_d2["location"].rsplit("/", 1)[-1]
    _, _, c2 = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/drafts/{d2_id}:propose", token=sergio, verbose=False)
    c2_id = c2["id"].rsplit("/", 1)[-1]
    lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/candidates/{c2_id}:approve", token=sergio, verbose=False)

    # 3. Wait for client to sync 2.0.0 and inspect GET /audit/reports/latest + JSONL events
    lab.wait_synced("2026-09-10T10:00:00Z")
    st, _, audit_rep = lab.request("GET", f"{lab.admin_url}/audit/reports/latest")
    assert st == 200 and audit_rep["version"] == "2.0.0"
    codes_seen = [e["code"] for e in audit_rep["entries"]]
    assert "AUDIT_WILL_WARN" in codes_seen and "AUDIT_WILL_REJECT" in codes_seen

    show_events_table(lab.read_events(), title="Client JSONL Events (Verdicts & Passive Audit)")
finally:
    lab.close()